In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
  pip install torch torchvision pillow tqdm


In [ ]:
"""
SKIN ANALYZER - DATASET DE PROBLÈMES DE PEAU (9,770 images)
Classification: Acné, Points noirs, Taches brunes, Pores, Rides
"""

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

DATASET_DIR = "/content/drive/MyDrive/deep learning/dataset/Skin v2"
MODEL_PATH = "skin_issues_best.pth"
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASSES = ['acne', 'blackheades', 'dark spots', 'pores', 'wrinkles']
NUM_CLASSES = 5

print("="*60)
print("ANALYSEUR DE PEAU - DATASET DE PROBLÈMES CUTANÉS")
print("="*60)
print(f"Périphérique: {DEVICE}")
print(f"Classes: {CLASSES}")
print(f"Chemin du dataset: {DATASET_DIR}")

ANALYSEUR DE PEAU - DATASET DE PROBLÈMES CUTANÉS
Périphérique: cpu
Classes: ['acne', 'blackheades', 'dark spots', 'pores', 'wrinkles']
Chemin du dataset: /content/drive/MyDrive/deep learning/dataset/Skin v2


In [ ]:
# ============================================================================
# TRANSFORMATIONS DES IMAGES
# ============================================================================

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# ============================================================================
# DATASET PERSONNALISÉ
# ============================================================================

class SkinDataset(Dataset):
    """
    Dataset personnalisé pour la séparation entraînement/validation.

    Args:
        dataset: Dataset ImageFolder complet (transform=None)
        indices: Liste des indices à conserver
        transform: Transformations à appliquer aux images
    """
    def __init__(self, dataset, indices, transform):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, label = self.dataset[self.indices[idx]]
        image = self.transform(image)
        return image, label

In [ ]:
# ============================================================================
# CHARGEMENT DES DONNÉES
# ============================================================================

print("\nChargement des images...")

full_dataset = datasets.ImageFolder(DATASET_DIR, transform=None)

train_idx, val_idx = train_test_split(
    range(len(full_dataset)),
    test_size=0.2,
    stratify=full_dataset.targets,
    random_state=42
)

train_dataset = SkinDataset(full_dataset, train_idx, train_transform)
val_dataset = SkinDataset(full_dataset, val_idx, val_transform)

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\nTotal des images: {len(full_dataset)}")
print(f"Entraînement: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")


from collections import Counter
counts = Counter(full_dataset.targets)
for i, cls in enumerate(CLASSES):
    print(f"{cls}: {counts[i]} images")


Chargement des images...

Total des images: 9770
Entraînement: 7816
Validation: 1954
acne: 2060 images
blackheades: 1970 images
dark spots: 2126 images
pores: 1632 images
wrinkles: 1982 images


In [ ]:
# ============================================================================
# CRÉATION DU MODÈLE
# ============================================================================

print("\nConstruction du modèle...")

model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

for param in model.features.parameters():
    param.requires_grad = False

in_features = model.classifier[0].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 256),
    nn.Hardswish(),
    nn.Dropout(0.2),
    nn.Linear(256, NUM_CLASSES)
)

model = model.to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Modèle chargé avec succès")
print(f"Paramètres totaux: {total_params:,}")
print(f"Paramètres entraînables: {trainable_params:,}")


Construction du modèle...
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 107MB/s]


Modèle chargé avec succès
Paramètres totaux: 1,076,005
Paramètres entraînables: 148,997


In [ ]:
# ============================================================================
# PRÉPARATION AVANT ENTRAÎNEMENT
# ============================================================================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

best_acc = 0
history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

print("\n" + "="*60)
print("DÉBUT DE L'ENTRAÎNEMENT")
print("="*60)


DÉBUT DE L'ENTRAÎNEMENT


In [ ]:
# ============================================================================
# BOUCLE D'ENTRAÎNEMENT
# ============================================================================

for epoch in range(EPOCHS):

    # PHASE D'ENTRAÎNEMENT
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for images, labels in loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = outputs.max(1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

        loop.set_postfix(loss=loss.item(), acc=100*train_correct/train_total)

    train_acc = 100 * train_correct / train_total
    train_loss_avg = train_loss / len(train_loader)

    # PHASE DE VALIDATION
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = outputs.max(1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100 * val_correct / val_total
    val_loss_avg = val_loss / len(val_loader)

    # POST-EPOCH
    scheduler.step(val_loss_avg)

    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_loss'].append(train_loss_avg)
    history['val_loss'].append(val_loss_avg)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"   Train - Loss: {train_loss_avg:.4f} | Acc: {train_acc:.2f}%")
    print(f"   Val   - Loss: {val_loss_avg:.4f} | Acc: {val_acc:.2f}%")
    print(f"   LR: {optimizer.param_groups[0]['lr']:.6f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'classes': CLASSES
        }, MODEL_PATH)
        print(f"   NOUVEAU RECORD! Sauvegardé ({best_acc:.2f}%)")

    print("-" * 60)

Epoch 1/30 [Val]: 100%|██████████| 31/31 [01:20<00:00,  2.59s/it]



Epoch 1/30
   Train - Loss: 0.7107 | Acc: 74.51%
   Val   - Loss: 0.4543 | Acc: 84.90%
   LR: 0.001000
   NOUVEAU RECORD! Sauvegardé (84.90%)
------------------------------------------------------------


Epoch 2/30 [Val]: 100%|██████████| 31/31 [00:37<00:00,  1.20s/it]



Epoch 2/30
   Train - Loss: 0.5019 | Acc: 82.19%
   Val   - Loss: 0.3718 | Acc: 87.82%
   LR: 0.001000
   NOUVEAU RECORD! Sauvegardé (87.82%)
------------------------------------------------------------


Epoch 3/30 [Val]: 100%|██████████| 31/31 [00:37<00:00,  1.21s/it]


...


Epoch 28/30
   Train - Loss: 0.1809 | Acc: 93.46%
   Val   - Loss: 0.1404 | Acc: 95.65%
   LR: 0.001000
------------------------------------------------------------


Epoch 29/30 [Val]: 100%|██████████| 31/31 [00:36<00:00,  1.18s/it]



Epoch 29/30
   Train - Loss: 0.1848 | Acc: 93.69%
   Val   - Loss: 0.1305 | Acc: 96.06%
   LR: 0.001000
   NOUVEAU RECORD! Sauvegardé (96.06%)
------------------------------------------------------------


Epoch 30/30 [Val]: 100%|██████████| 31/31 [00:39<00:00,  1.27s/it]


Epoch 30/30
   Train - Loss: 0.1811 | Acc: 93.39%
   Val   - Loss: 0.1357 | Acc: 95.96%
   LR: 0.001000
------------------------------------------------------------


In [ ]:
# ============================================================================
# RAPPORT FINAL
# ============================================================================

print("\n" + "="*60)
print("RÉSULTATS FINAUX")
print("="*60)
print(f"Meilleure précision en validation: {best_acc:.2f}%")

checkpoint = torch.load(MODEL_PATH)
model.load_state_dict(checkpoint['model_state_dict'])

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("\nRapport de classification:")
print(classification_report(all_labels, all_preds, target_names=CLASSES))

print("\nMatrice de confusion:")
print(confusion_matrix(all_labels, all_preds))


RÉSULTATS FINAUX
Meilleure précision en validation: 96.06%

Rapport de classification:
              precision    recall  f1-score   support

        acne       0.96      0.93      0.95       412
 blackheades       0.93      0.94      0.93       394
  dark spots       0.98      0.96      0.97       425
       pores       0.98      0.99      0.98       327
    wrinkles       0.96      0.99      0.97       396

    accuracy                           0.96      1954
   macro avg       0.96      0.96      0.96      1954
weighted avg       0.96      0.96      0.96      1954


Matrice de confusion:
[[384  19   5   1   3]
 [ 13 369   3   2   7]
 [  2   6 408   1   8]
 [  0   3   1 323   0]
 [  1   0   0   2 393]]


In [ ]:
from google.colab import files
files.download(MODEL_PATH)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>